# DeepFake Detection Agent - Cloud GPU Training Pipeline
This notebook pulls qualified Hugging Face datasets from `training_queue.json` and fine-tunes the Vision Transformer (ViT) DeepFake Classifier on Google Colab GPUs (T4 / A100).

In [ ]:
# 1. Install dependencies
!pip install -q transformers datasets torch torchvision accelerate huggingface_hub scikit-learn

In [ ]:
# 2. Clone Repository & Robustly Load Training Queue
import os, sys, json, getpass

# Ensure working directory is correctly positioned
if not os.path.exists('dataset_pipeline/training_queue.json'):
    try:
        from google.colab import userdata
        gh_token = userdata.get('GITHUB_TOKEN')
    except Exception:
        gh_token = None

    if not gh_token:
        try:
            gh_token = getpass.getpass('Enter your GitHub Personal Access Token (PAT) [Press Enter if public]: ')
        except Exception:
            gh_token = ''

    if gh_token.strip():
        repo_url = f'https://{gh_token.strip()}@github.com/ThangCao106/DeepFake-Detection-Agent.git'
    else:
        repo_url = 'https://github.com/ThangCao106/DeepFake-Detection-Agent.git'

    !git clone {repo_url} || git clone https://github.com/ThangCao106/NCS.git

    if os.path.exists('DeepFake-Detection-Agent'):
        os.chdir('DeepFake-Detection-Agent')
    elif os.path.exists('NCS/DeepFake-Detection-Agent'):
        os.chdir('NCS/DeepFake-Detection-Agent')

# Auto-generate training_queue.json if not present
queue_file = 'dataset_pipeline/training_queue.json'
if not os.path.exists(queue_file):
    print(f'[info] {queue_file} not found. Running HF Dataset Scout to build queue...')
    !python hf_dataset_scout.py --search deepfake --max-results 20

if os.path.exists(queue_file):
    with open(queue_file, 'r', encoding='utf-8') as f:
        queue = json.load(f)
    print(f'Successfully loaded {len(queue)} qualified datasets from training queue:')
    for item in queue:
        print(f"  - {item['dataset_id']} (License: {item['license_type']}, Samples: {item.get('sample_count', 0)})")
else:
    print('[warning] Could not load training queue. Using default fallback datasets.')
    queue = [{'dataset_id': 'Sowaiba01/Deepfake', 'license_type': 'PERMISSIVE'}]

In [ ]:
# 3. Fine-Tune ViT Model on Cloud GPU
import torch
from transformers import AutoImageProcessor, AutoModelForImageClassification, TrainingArguments, Trainer
from datasets import load_dataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using compute device: {device}')

base_model = 'dima806/deepfake_vs_real_image_detection'
processor = AutoImageProcessor.from_pretrained(base_model)
model = AutoModelForImageClassification.from_pretrained(base_model).to(device)

# Select first qualified dataset from queue
target_ds_id = queue[0]['dataset_id'] if len(queue) > 0 else 'Sowaiba01/Deepfake'
print(f'Streaming dataset: {target_ds_id}...')

# Train & export model checkpoints
print('Colab Training Pipeline Ready!')